# Desafío Random Forest — Regresión

**Objetivo:** desarrollar un modelo de Random Forest de regresión para predecir el precio de venta de inmuebles de Ames, Iowa, usando como variable objetivo `Sale_Price`.

> **Nota importante:** el enunciado solicita usar la base de Ames y archivos serializados de la sesión anterior. Este notebook está preparado para funcionar si en la misma carpeta se agregan archivos como `ames.csv`, `AmesHousing.csv`, `X_train.pkl`, `X_test.pkl`, `y_train.pkl`, `y_test.pkl` y/o un modelo `.pkl`. Si esos archivos no están disponibles, el notebook lo informará sin inventar resultados.

## Ejercicio 1 — Preparación del ambiente de trabajo

Se importan las librerías clásicas de análisis de datos, visualización, modelos de Random Forest y métricas para regresión. Además, se define una carga flexible de la base, eliminando `Unnamed: 0` si aparece.

In [1]:
# Librerías clásicas
import os
import glob
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Preprocesamiento y modelamiento
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# Métricas de regresión
from sklearn.metrics import mean_squared_error, median_absolute_error, r2_score

warnings.filterwarnings("ignore")
%matplotlib inline

RANDOM_STATE = 11238
plt.rcParams["figure.figsize"] = (10, 6)

In [2]:
def buscar_base_ames():
    """Busca automáticamente una base de Ames con la variable Sale_Price."""
    posibles_archivos = [
        "ames.csv", "AmesHousing.csv", "ames_housing.csv", "house_prices.csv",
        "train.csv", "ames_train.csv", "Ames.csv"
    ]
    candidatos = []
    for archivo in posibles_archivos:
        if os.path.exists(archivo):
            candidatos.append(archivo)
    candidatos.extend(glob.glob("*.csv"))
    candidatos = list(dict.fromkeys(candidatos))

    for archivo in candidatos:
        try:
            tmp = pd.read_csv(archivo)
            columnas_limpias = [c.replace("\ufeff", "").strip() for c in tmp.columns]
            tmp.columns = columnas_limpias
            if "Sale_Price" in tmp.columns:
                print(f"Base encontrada: {archivo}")
                return tmp, archivo
        except Exception:
            pass

    raise FileNotFoundError(
        "No se encontró una base CSV con la variable 'Sale_Price'. "
        "Agrega la base de Ames en la misma carpeta, por ejemplo como 'ames.csv' o 'AmesHousing.csv'."
    )

try:
    df, archivo_base = buscar_base_ames()
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns="Unnamed: 0")
    print(df.shape)
    display(df.head())
except FileNotFoundError as e:
    df = None
    print(e)

No se encontró una base CSV con la variable 'Sale_Price'. Agrega la base de Ames en la misma carpeta, por ejemplo como 'ames.csv' o 'AmesHousing.csv'.


In [3]:
# Medidas descriptivas generales de la base
if df is not None:
    display(df.describe(include="all").T)
else:
    print("Pendiente: cargar la base Ames con la variable Sale_Price.")

Pendiente: cargar la base Ames con la variable Sale_Price.


### Preparación de matriz de atributos y vector objetivo

Como `sklearn` no puede entrenar directamente con variables categóricas en formato texto, se crea un `Pipeline` que:

1. Imputa valores perdidos en variables numéricas con la mediana.
2. Imputa valores perdidos en variables categóricas con la categoría más frecuente.
3. Binariza variables categóricas mediante `OneHotEncoder`.
4. Entrena el modelo de regresión.

Esta decisión reemplaza el preprocesamiento manual cuando no se encuentran los archivos serializados de la sesión anterior.

In [4]:
def construir_preprocesador(X):
    """Construye un ColumnTransformer para variables numéricas y categóricas."""
    numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
    categorical_features = X.select_dtypes(exclude=["int64", "float64", "int32", "float32"]).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", encoder)
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ],
        remainder="drop"
    )
    return preprocessor, numeric_features, categorical_features

if df is not None:
    y = df["Sale_Price"]
    X = df.drop(columns="Sale_Price")

    # Eliminamos columnas de identificación si existen, porque no aportan información predictiva real.
    posibles_id = ["PID", "Order", "Id", "ID"]
    X = X.drop(columns=[c for c in posibles_id if c in X.columns], errors="ignore")

    preprocessor, numeric_features, categorical_features = construir_preprocesador(X)

    print(f"Variables numéricas: {len(numeric_features)}")
    print(f"Variables categóricas: {len(categorical_features)}")
else:
    X = y = preprocessor = None
    print("Pendiente: no se puede construir X/y sin la base Ames.")

Pendiente: no se puede construir X/y sin la base Ames.


## Ejercicio 2 — Importación de archivos serializados

Se intenta importar automáticamente modelos y conjuntos serializados (`.pkl`) de la sesión anterior. Si no existen, se genera una segmentación de entrenamiento/validación desde la base original.

In [5]:
def cargar_pickle_posible(nombres):
    """Carga el primer archivo pickle existente dentro de una lista de nombres posibles."""
    for nombre in nombres:
        if os.path.exists(nombre):
            with open(nombre, "rb") as f:
                print(f"Archivo serializado cargado: {nombre}")
                return pickle.load(f), nombre
    return None, None

# Posibles nombres habituales de archivos serializados
X_train_serial, xtrain_name = cargar_pickle_posible(["X_train.pkl", "x_train.pkl", "X_train.sav", "xtrain.pkl"])
X_test_serial, xtest_name = cargar_pickle_posible(["X_test.pkl", "x_test.pkl", "X_test.sav", "xtest.pkl"])
y_train_serial, ytrain_name = cargar_pickle_posible(["y_train.pkl", "y_train.sav", "ytrain.pkl"])
y_test_serial, ytest_name = cargar_pickle_posible(["y_test.pkl", "y_test.sav", "ytest.pkl"])
modelo_companero, model_name = cargar_pickle_posible([
    "modelo_companero.pkl", "modelo.pkl", "model.pkl", "regression_model.pkl",
    "linear_model.pkl", "decision_tree_model.pkl"
])

serializados_completos = all(obj is not None for obj in [X_train_serial, X_test_serial, y_train_serial, y_test_serial])

if serializados_completos:
    print("Se usarán los conjuntos serializados encontrados.")
    X_train, X_test = X_train_serial, X_test_serial
    y_train, y_test = y_train_serial, y_test_serial
    usar_pipeline = False
elif df is not None:
    print("No se encontraron todos los archivos serializados. Se crearán conjuntos train/test desde la base Ames.")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.33, random_state=RANDOM_STATE
    )
    usar_pipeline = True
else:
    X_train = X_test = y_train = y_test = None
    usar_pipeline = None
    print("Pendiente: se requiere base Ames o archivos serializados completos.")

Pendiente: se requiere base Ames o archivos serializados completos.


In [6]:
def report_scores(y_true, y_pred, nombre_modelo="Modelo"):
    """Imprime métricas clásicas para regresión."""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    medae = median_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"{nombre_modelo}")
    print("-" * len(nombre_modelo))
    print(f"MSE   : {mse:,.4f}")
    print(f"RMSE  : {rmse:,.4f}")
    print(f"MedAE : {medae:,.4f}")
    print(f"R2    : {r2:,.4f}")

    return {"modelo": nombre_modelo, "mse": mse, "rmse": rmse, "medae": medae, "r2": r2}

In [7]:
resultados = []

if modelo_companero is not None and X_test is not None:
    try:
        pred_companero = modelo_companero.predict(X_test)
        resultados.append(report_scores(y_test, pred_companero, "Modelo serializado / compañero"))
    except Exception as error:
        print("No se pudo ejecutar el modelo serializado.")
        print("Causas probables:")
        print("1. El modelo fue entrenado con otras columnas o en otro orden.")
        print("2. El modelo espera datos ya preprocesados y aquí están en formato original.")
        print("3. Hay incompatibilidad de versiones entre sklearn/pandas.")
        print("4. Falta cargar el preprocesador usado originalmente.")
        print(f"Detalle técnico: {error}")
else:
    print("No se encontró un modelo serializado para comparar.")

No se encontró un modelo serializado para comparar.


## Ejercicio 3 — Evaluación Random Forest

Se entrena un modelo `RandomForestRegressor` sin modificar hiperparámetros centrales como `n_estimators`, `max_depth` o `max_features`. Sólo se declara la semilla pseudoaleatoria y `oob_score=True` para obtener una referencia out-of-bag.

In [8]:
if X_train is not None:
    if usar_pipeline:
        rf_default = Pipeline(steps=[
            ("preprocess", preprocessor),
            ("model", RandomForestRegressor(
                random_state=RANDOM_STATE,
                oob_score=True,
                n_jobs=-1
            ))
        ])
    else:
        rf_default = RandomForestRegressor(
            random_state=RANDOM_STATE,
            oob_score=True,
            n_jobs=-1
        )

    rf_default.fit(X_train, y_train)
    pred_rf_default = rf_default.predict(X_test)
    resultados.append(report_scores(y_test, pred_rf_default, "Random Forest por defecto"))

    try:
        oob_default = rf_default.named_steps["model"].oob_score_ if usar_pipeline else rf_default.oob_score_
        print(f"OOB R2 Random Forest por defecto: {oob_default:,.4f}")
    except Exception as error:
        print(f"No fue posible obtener OOB score: {error}")
else:
    print("Pendiente: faltan datos para entrenar el Random Forest.")

Pendiente: faltan datos para entrenar el Random Forest.


### Comparación breve

El mejor modelo será el que presente **menor RMSE/MSE** y **mayor R²** en la muestra de validación. Si no se pudo cargar el modelo serializado de la sesión anterior o el de un compañero, la comparación queda limitada al Random Forest entrenado en este notebook.

In [9]:
if len(resultados) > 0:
    resultados_df = pd.DataFrame(resultados).sort_values("rmse")
    display(resultados_df)
else:
    print("No hay métricas disponibles todavía.")

No hay métricas disponibles todavía.


## Ejercicio 4 — Evaluación de hiperparámetros

Se evalúan las combinaciones solicitadas:

- `max_features`: `None`, `log2`, `sqrt`.
- `n_estimators`: entre 20 y 1000, en pasos de 50.

Para cada combinación se guarda:

- `oob_r2`: desempeño out-of-bag.
- `oob_error`: calculado como `1 - oob_r2`.
- `val_rmse`: error RMSE en la muestra de validación.
- `val_r2`: R² en la muestra de validación.

In [10]:
if X_train is not None:
    max_features_grid = [None, "log2", "sqrt"]
    n_estimators_grid = list(range(20, 1000, 50)) + [1000]
    registros = []

    for max_features in max_features_grid:
        for n_estimators in n_estimators_grid:
            if usar_pipeline:
                modelo = Pipeline(steps=[
                    ("preprocess", preprocessor),
                    ("model", RandomForestRegressor(
                        n_estimators=n_estimators,
                        max_features=max_features,
                        oob_score=True,
                        random_state=RANDOM_STATE,
                        n_jobs=-1
                    ))
                ])
            else:
                modelo = RandomForestRegressor(
                    n_estimators=n_estimators,
                    max_features=max_features,
                    oob_score=True,
                    random_state=RANDOM_STATE,
                    n_jobs=-1
                )

            modelo.fit(X_train, y_train)
            y_pred = modelo.predict(X_test)

            fitted_rf = modelo.named_steps["model"] if usar_pipeline else modelo
            oob_r2 = fitted_rf.oob_score_
            val_mse = mean_squared_error(y_test, y_pred)
            val_rmse = np.sqrt(val_mse)
            val_r2 = r2_score(y_test, y_pred)

            registros.append({
                "max_features": str(max_features),
                "max_features_param": max_features,
                "n_estimators": n_estimators,
                "oob_r2": oob_r2,
                "oob_error": 1 - oob_r2,
                "val_rmse": val_rmse,
                "val_r2": val_r2
            })

    rf_grid_results = pd.DataFrame(registros)
    display(rf_grid_results.sort_values("oob_error").head(10))
else:
    rf_grid_results = None
    print("Pendiente: faltan datos para evaluar hiperparámetros.")

Pendiente: faltan datos para evaluar hiperparámetros.


In [11]:
if rf_grid_results is not None:
    plt.figure(figsize=(12, 6))
    for max_features in rf_grid_results["max_features"].unique():
        tmp = rf_grid_results[rf_grid_results["max_features"] == max_features]
        plt.plot(tmp["n_estimators"], tmp["oob_error"], marker="o", label=f"max_features={max_features}")

    plt.title("Evolución del error OOB según número de árboles")
    plt.xlabel("n_estimators")
    plt.ylabel("OOB error = 1 - OOB R2")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No hay resultados para graficar.")

No hay resultados para graficar.


In [12]:
if rf_grid_results is not None:
    plt.figure(figsize=(12, 6))
    for max_features in rf_grid_results["max_features"].unique():
        tmp = rf_grid_results[rf_grid_results["max_features"] == max_features]
        plt.plot(tmp["n_estimators"], tmp["val_rmse"], marker="o", label=f"max_features={max_features}")

    plt.title("RMSE de validación según número de árboles")
    plt.xlabel("n_estimators")
    plt.ylabel("RMSE validación")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No hay resultados para graficar.")

No hay resultados para graficar.


## Ejercicio 5 — Refactorización del modelo

Se selecciona la combinación con menor `oob_error`, se reentrena el modelo final y se reportan sus métricas en validación.

In [13]:
if rf_grid_results is not None:
    mejor = rf_grid_results.sort_values("oob_error").iloc[0]
    best_max_features = mejor["max_features_param"]
    best_n_estimators = int(mejor["n_estimators"])

    print("Mejor combinación según OOB:")
    print(f"max_features : {best_max_features}")
    print(f"n_estimators : {best_n_estimators}")
    print(f"OOB R2       : {mejor['oob_r2']:.4f}")
    print(f"OOB error    : {mejor['oob_error']:.4f}")

    if usar_pipeline:
        rf_final = Pipeline(steps=[
            ("preprocess", preprocessor),
            ("model", RandomForestRegressor(
                n_estimators=best_n_estimators,
                max_features=best_max_features,
                oob_score=True,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])
    else:
        rf_final = RandomForestRegressor(
            n_estimators=best_n_estimators,
            max_features=best_max_features,
            oob_score=True,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    rf_final.fit(X_train, y_train)
    pred_final = rf_final.predict(X_test)
    resultados.append(report_scores(y_test, pred_final, "Random Forest refactorizado"))
else:
    rf_final = None
    print("Pendiente: primero se debe ejecutar la evaluación de hiperparámetros.")

Pendiente: primero se debe ejecutar la evaluación de hiperparámetros.


### Comentario esperado sobre desempeño

Al ejecutar el notebook con la base de Ames, el modelo refactorizado debe compararse con el Random Forest por defecto. La mejora o empeoramiento se evalúa observando:

- Si baja el `RMSE`, el error promedio de predicción disminuye.
- Si sube el `R2`, el modelo explica una mayor proporción de la variabilidad de `Sale_Price`.
- Si el `OOB R2` es consistente con el `R2` de validación, el modelo tiene un comportamiento más estable.

En Random Forest, aumentar `n_estimators` suele estabilizar el desempeño, pero después de cierto punto la mejora marginal se vuelve pequeña. La elección de `max_features` controla la aleatoriedad entre árboles: `None` se parece más a bagging, mientras que `sqrt` y `log2` introducen más diversidad en el ensamble.

## Inspección adicional — Importancia de atributos

Aunque Random Forest es menos interpretable que un modelo lineal, permite revisar la importancia relativa de cada atributo en la reducción de impureza de los árboles.

In [14]:
def obtener_nombres_features(preprocessor, X_original):
    """Obtiene nombres de variables luego del preprocesamiento."""
    try:
        return preprocessor.get_feature_names_out()
    except Exception:
        return np.array(X_original.columns)

if rf_final is not None:
    if usar_pipeline:
        modelo_rf = rf_final.named_steps["model"]
        nombres = obtener_nombres_features(rf_final.named_steps["preprocess"], X_train)
    else:
        modelo_rf = rf_final
        nombres = np.array(X_train.columns) if hasattr(X_train, "columns") else np.array([f"x{i}" for i in range(X_train.shape[1])])

    importancias = pd.DataFrame({
        "feature": nombres,
        "importance": modelo_rf.feature_importances_
    }).sort_values("importance", ascending=False)

    display(importancias.head(20))

    plt.figure(figsize=(10, 8))
    top = importancias.head(20).sort_values("importance")
    plt.barh(top["feature"], top["importance"])
    plt.title("Top 20 atributos más importantes")
    plt.xlabel("Importancia")
    plt.ylabel("Atributo")
    plt.show()
else:
    print("Pendiente: entrenar el modelo final.")

Pendiente: entrenar el modelo final.


## Conclusión

Este notebook implementa el flujo completo solicitado para Random Forest de regresión: carga de datos, uso opcional de archivos serializados, segmentación de entrenamiento/validación, entrenamiento de un Random Forest por defecto, búsqueda de hiperparámetros mediante OOB y reentrenamiento del modelo final.

Si los archivos serializados de la sesión anterior no están disponibles, no es posible comparar empíricamente contra el modelo propio previo ni contra el modelo de un compañero. En ese caso, la causa principal no es metodológica, sino de disponibilidad de archivos: el modelo serializado puede requerir exactamente las mismas columnas, orden de variables, transformaciones y versión de librerías con que fue entrenado originalmente.